In [1]:
import pandas as pd
import numpy as np


In [2]:
rfm_data = pd.read_csv('data/output_query/raw_rfm.csv', index_col='customer_id')

In [3]:
rfm_data.info()

<class 'pandas.DataFrame'>
Index: 27775 entries, 42711 to 44362
Data columns (total 3 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   recency    27775 non-null  int64  
 1   frequency  27775 non-null  int64  
 2   monetary   27775 non-null  float64
dtypes: float64(1), int64(2)
memory usage: 868.0 KB


In [4]:
rfm_data.head()

,recency,frequency,monetary
customer_id,,,
42711,1083,1,67.989998
48284,778,1,29.990000
82244,449,1,8.500000
65264,885,1,25.000000
14324,556,1,5.990000


In [5]:
rfm_data.describe()

,recency,frequency,monetary
count,27775.000000,27775.000000,27775.000000
mean,460.415878,1.131521,98.266912
std,426.751394,0.374752,105.231385
min,1.000000,1.000000,0.490000
25%,111.000000,1.000000,32.889999
50%,328.000000,1.000000,63.740002
75%,705.000000,1.000000,128.000000
max,1920.000000,4.000000,1221.619995


In [6]:
def f_score_rule(freq):
    if freq == 1:
        return 1
    elif freq == 2:
        return 2
    else:
        return 3
    

In [7]:
rfm_data['F_score'] = rfm_data['frequency'].apply(f_score_rule)

In [8]:
rfm_data['R_score'] = pd.qcut(
    rfm_data['recency'],
    q=5,
    labels = [5, 4, 3, 2, 1]
)
rfm_data['M_score'] = pd.qcut(
    rfm_data['monetary'],
    q=5,
    labels = [1, 2, 3, 4, 5]
)

In [9]:
rfm_data.head()

,recency,frequency,monetary,F_score,R_score,M_score
customer_id,,,,,,
42711,1083,1,67.989998,1,1,3
48284,778,1,29.990000,1,2,2
82244,449,1,8.500000,1,3,1
65264,885,1,25.000000,1,1,1
14324,556,1,5.990000,1,2,1


In [10]:
rfm_data['RFM_score'] = rfm_data['R_score'].astype(str) + rfm_data['F_score'].astype(str) + rfm_data['M_score'].astype(str)

In [11]:
rfm_data

,recency,frequency,monetary,F_score,R_score,M_score,RFM_score
customer_id,,,,,,,
42711,1083,1,67.989998,1,1,3,113
48284,778,1,29.990000,1,2,2,212
82244,449,1,8.500000,1,3,1,311
65264,885,1,25.000000,1,1,1,111
14324,556,1,5.990000,1,2,1,211
...,...,...,...,...,...,...,...
29214,93,1,62.000000,1,4,3,413
28132,40,1,76.990002,1,5,3,513
43280,226,1,12.000000,1,4,1,411


In [12]:
rfm_data['R_score'] = rfm_data['R_score'].astype(int)
rfm_data['M_score'] = rfm_data['M_score'].astype(int)

In [13]:
rfm_data[['R_score', 'F_score', 'M_score']].describe()

,R_score,F_score,M_score
count,27775.000000,27775.000000,27775.000000
mean,3.002736,1.130837,2.999856
std,1.415000,0.370403,1.414392
min,1.000000,1.000000,1.000000
25%,2.000000,1.000000,2.000000
50%,3.000000,1.000000,3.000000
75%,4.000000,1.000000,4.000000
max,5.000000,3.000000,5.000000


Ngưỡng Recency trong phân tích này khá cao (P25=111 ngày, P75=705 ngày) so với thông lệ ngành eCommerce (thường dùng 30-90 ngày). Điều này phản ánh đặc điểm của TheLook: phần lớn khách hàng có chu kỳ quay lại rất dài hoặc không quay lại — càng củng cố insight về vấn đề retention nghiêm trọng đã phát hiện ở phần Frequency (88% one-time buyers)

#### Customer Segments

In [14]:
# Recency thresholds
p25 = 111
median = 328
p75 = 705

# Monetary threshold
m_threshold = rfm_data['monetary'].quantile(0.75) # M cao là nhóm thiểu số nổi bật, nhóm có 25% giá trị cao nhất

def assign_segment(data):
    if data['frequency'] >=3:
        if data['recency'] <= median:
            if data['monetary'] >= m_threshold:
                data['segment'] = 'Champions'
            else: 
                data['segment'] = 'Loyal Buyers'
        else:
            data['segment'] = 'At Risk'
    elif data['frequency'] == 2:
        data['segment'] = 'Repeat Buyers'
    elif data['frequency'] == 1:
        if data['recency'] <= p25:
            if data['monetary'] >= m_threshold:
                data['segment'] = 'High-Value New'
            else:
                data['segment'] = 'New Customers'
        elif data['recency'] <= p75:
            data['segment'] = 'Dormant'
        elif data['recency'] > p75:
            data['segment'] = 'Lost'
    return data

In [15]:
rfm_data = rfm_data.apply(assign_segment, axis=1)

In [16]:
rfm_data['segment'].value_counts()
rfm_data['segment'].value_counts(normalize=True) * 100

segment
Dormant           43.751575
Lost              23.506751
New Customers     16.457246
Repeat Buyers     10.736274
High-Value New     4.374437
Champions          0.669667
At Risk            0.302430
Loyal Buyers       0.201620
Name: proportion, dtype: float64

Do các nhóm F>=3 chiếm số lượng quá nhỏ -> gộp thành 1 nhóm

In [17]:
# Recency thresholds
p25 = 111
median = 328
p75 = 705

# Monetary threshold
m_threshold = rfm_data['monetary'].quantile(0.75) # M cao là nhóm thiểu số nổi bật, nhóm có 25% giá trị cao nhất

def assign_segment(data):
    if data['frequency'] >=3:
        data['segment'] = 'Loyal Customers'
    elif data['frequency'] == 2:
        data['segment'] = 'Repeat Buyers'
    elif data['frequency'] == 1:
        if data['recency'] <= p25:
            if data['monetary'] >= m_threshold:
                data['segment'] = 'High-Value New'
            else:
                data['segment'] = 'New Customers'
        elif data['recency'] <= p75:
            data['segment'] = 'Inactive Customers'
        elif data['recency'] > p75:
            data['segment'] = 'Lost'
    return data

In [18]:
rfm_data = rfm_data.apply(assign_segment, axis=1)

In [19]:
rfm_data['segment'].value_counts()

segment
Inactive Customers    12152
Lost                   6529
New Customers          4571
Repeat Buyers          2982
High-Value New         1215
Loyal Customers         326
Name: count, dtype: int64

In [20]:
rfm_data['segment'].value_counts(normalize=True) * 100

segment
Inactive Customers    43.751575
Lost                  23.506751
New Customers         16.457246
Repeat Buyers         10.736274
High-Value New         4.374437
Loyal Customers        1.173717
Name: proportion, dtype: float64

In [21]:
rfm_data.info()

<class 'pandas.DataFrame'>
Index: 27775 entries, 42711 to 44362
Data columns (total 8 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   recency    27775 non-null  int64  
 1   frequency  27775 non-null  int64  
 2   monetary   27775 non-null  float64
 3   F_score    27775 non-null  int64  
 4   R_score    27775 non-null  int64  
 5   M_score    27775 non-null  int64  
 6   RFM_score  27775 non-null  str    
 7   segment    27775 non-null  str    
dtypes: float64(1), int64(5), str(2)
memory usage: 2.3 MB


In [22]:
rfm_data.to_csv('data/output_query/rfm_scored.csv')